
# 03 — Dual-Branch PERFECT (Wordlist-Focused, 4M, Kaggle T4 16GB)

> **Goal:** maximize **val-unseen wordlist macro-recall** (`suppobox/bigviktor/manuelita`) — the core research question `CSE427_DGA_MASTER_PLAN.md:13`. This is binary `DGA vs benign` (`legit` 1:1), per-family recall is post-hoc slice of binary predictions. `CF` `2×2` is correct.
> **Why past runs stayed 0.27:** `wordninja → mean([FastText])` (`training/03_dual_branch_kaggle_fp32.ipynb:199`) washed out word-word coherence. Real FastText alone gave `+0.03`. Need **word-sequence** + **gated fusion**.
> **Hardware:** T4 16GB — `FP32` `batch 64 + grad_accum 2 = 128` `4.1M` params `dropout 0.5` `WD 5e-4` `12 epochs`.

> **Kaggle:** Input `/kaggle/input/dga-umudga-rhena-merged` or `/kaggle/input/datasets/walidmahmood/dga-umudga-rhena-merged`, Working `/kaggle/working` (outputs survive Save Version). Needs **Internet ON** for `cc.en.300.bin` if not added as dataset `cc-en-300-bin`.

> **Local:** `G:\dga_killer` fallback.


In [ ]:

# Kaggle T4 — install once (uncomment if needed on fresh session)
!pip install -q wordninja fasttext tldextract scikit-learn seaborn

import pathlib, json, math, random, os, glob, subprocess, sys
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve, average_precision_score, f1_score, accuracy_score
try:
    import tldextract
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tldextract"])
    import tldextract

# --- Path auto-detect: Kaggle vs Local — robust for /kaggle/input/datasets/... ---
import glob as _glob
kaggle_candidates = [
    Path("/kaggle/input/dga-umudga-rhena-merged"),
    Path("/kaggle/input/datasets/walidmahmood/dga-umudga-rhena-merged"),
] + [Path(p) for p in _glob.glob("/kaggle/input/*dga*")] + [Path(p) for p in _glob.glob("/kaggle/input/datasets/*/*dga*")]
try:
    print(f"/kaggle/input contents: {list(Path('/kaggle/input').glob('*'))[:10] if Path('/kaggle/input').exists() else 'no /kaggle/input'}")
except: pass
BASE = None
for cand in kaggle_candidates:
    if (cand / "train.csv").exists():
        BASE = cand
        break
if BASE is not None:
    WORKING = Path("/kaggle/working")
    print(f"Kaggle detected — BASE={BASE}")
    DATA_PROC = BASE
else:
    BASE = Path(r"G:\dga_killer")
    DATA_PROC = BASE / "data/processed"
    WORKING = BASE / "training"
    print(f"Local detected — BASE={BASE}")
if (BASE / "train.csv").exists():
    DATA_PROC = BASE
else:
    DATA_PROC = BASE / "data/processed"
FIG_DIR = WORKING / "training_figures" if "kaggle" in str(WORKING) else BASE / "training/figures"
WEIGHTS_DIR = WORKING / "training_weights" if "kaggle" in str(WORKING) else BASE / "training/weights"
FIG_DIR.mkdir(parents=True, exist_ok=True); WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"DATA_PROC={DATA_PROC} FIG_DIR={FIG_DIR} WEIGHTS_DIR={WEIGHTS_DIR}")
print(f"Files in DATA_PROC: {list(DATA_PROC.glob('*.csv'))[:5] if DATA_PROC.exists() else 'missing'}")
# VOCAB resolve
if (BASE / "train.csv").exists():
    VOCAB_PATH = BASE / "char_vocab.json"
    if not VOCAB_PATH.exists():
        VOCAB_PATH = BASE / "char_vocab.json"
else:
    VOCAB_PATH = BASE / "data/vocab/char_vocab.json"
if not VOCAB_PATH.exists():
    VOCAB_PATH = Path(r"G:\dga_killer/data/vocab/char_vocab.json")
print(f"VOCAB_PATH={VOCAB_PATH} exists={VOCAB_PATH.exists()}")

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device="cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device} — {torch.cuda.get_device_name(0) if device=='cuda' else 'CPU'}")
if device=="cuda":
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB — FP32 4.1M batch 64+accum2")


In [ ]:

# Load processed SLDs (lowercased, TLD-stripped) — Kaggle or Local
train_path = DATA_PROC / "train.csv"
val_path = DATA_PROC / "val_unseen.csv"
if not train_path.exists():
    train_path = BASE / "train.csv"
if not val_path.exists():
    val_path = BASE / "val_unseen.csv"
train = pd.read_csv(train_path)
val = pd.read_csv(val_path)
print(f"Train {len(train):,} Val {len(val):,}")
print("Train families:", sorted(train['family'].unique())[:10], "...", len(train['family'].unique()))
print("Train wordlist families:", sorted(train[train['wordlist']]['family'].unique()), "count", int(train['wordlist'].sum()))
print("Val families:", sorted(val['family'].unique()))
print(train["family"].value_counts().head())
# Inspect overlap warning (legit overlap is expected, wordlist overlap is leakage)
train_slds=set(train['sld'].astype(str))
val_slds=set(val['sld'].astype(str))
overlap=len(train_slds & val_slds)
print(f"SLD overlap train-val: {overlap} unique SLDs (legit overlap ~1236 is expected from top-domains list; wordlist overlap should be ~0)")
# Vocab (train-only)
vocab = json.load(open(VOCAB_PATH))
char2idx = vocab["vocab"]
vocab_size = vocab["size"]
print(f"Vocab {vocab_size} chars: {''.join(vocab['chars'][:20])}... + specials")
# FastText .bin — check Kaggle dataset inputs first, then auto-download if Internet ON
ft_bin = Path("/kaggle/working/cc.en.300.bin") if "kaggle" in str(BASE) else BASE / "cc.en.300.bin"
if not ft_bin.exists() and (BASE / "cc.en.300.bin").exists():
    ft_bin = BASE / "cc.en.300.bin"
print(f"FastText .bin initial path: {ft_bin} exists={ft_bin.exists()}")
import glob as _ft_glob
for cand in [Path("/kaggle/input/cc-en-300-bin/cc.en.300.bin"), Path("/kaggle/input/cc.en.300.bin"), Path("/kaggle/input/fasttext/cc.en.300.bin")] + [Path(x) for x in _ft_glob.glob("/kaggle/input/**/cc.en.300.bin")]:
    if cand.exists():
        ft_bin = cand
        print(f"Found FastText .bin in Kaggle input: {ft_bin}")
        break
if not ft_bin.exists():
    import urllib.request
    try:
        urllib.request.urlopen("https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz", timeout=5)
        _has_internet=True
    except:
        _has_internet=False
    if _has_internet:
        print("FastText .bin not found — downloading 4.2GB (8min, needs Internet ON)...")
        dl_path = Path("/kaggle/working/cc.en.300.bin.gz")
        bin_path = Path("/kaggle/working/cc.en.300.bin")
        if not bin_path.exists():
            try:
                subprocess.check_call(["wget", "-O", str(dl_path), "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz"])
                print(f"Downloaded {dl_path.stat().st_size/1e9:.2f}GB, gunzipping...")
                subprocess.check_call(["gunzip", "-f", str(dl_path)])
                print(f"Gunzip done: {bin_path.exists()} {bin_path.stat().st_size/1e9:.2f}GB" if bin_path.exists() else "Gunzip failed")
                ft_bin = bin_path
            except Exception as e:
                print(f"Download failed: {e} — will fallback to random (wordlist 0.27)")
        else:
            ft_bin = bin_path
    else:
        print("No internet and no .bin dataset — fallback random (add dataset cc-en-300-bin or enable Internet)")
if not ft_bin.exists():
    print("WARNING: Using random fallback — wordlist recall will stay ~0.27. For 0.55+ you MUST have real FastText.")
else:
    print(f"FastText .bin ready: {ft_bin} {ft_bin.stat().st_size/1e9:.2f}GB")



## 1. Datasets — Char + Wordninja + Frozen FastText (SEQUENCE, not mean)

**Fixes vs old:** `wordninja.split(sld) → [vec1, vec2, ...]` kept as **sequence** (not `mean`), so `that+merciless+pretended` coherence is preserved. Char `encode_chars` guards `NaN/empty → [PAD],1` to avoid `pack_padded_sequence` crash on `len 0`. `DGADataset(augment=True)` adds `5% char dropout` to prevent memorizing train families. `wordlist weight 6.0` (was `4.2`) forces model to see wordlist `1:1` per batch.


In [ ]:

try:
    import wordninja
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "wordninja"])
    import wordninja
FT_DIM = 300
try:
    import fasttext
    if ft_bin.exists():
        ft = fasttext.load_model(str(ft_bin))
        print("Loaded FastText .bin (frozen, subword fallback)")
        def get_vec(w): return ft.get_word_vector(w)
    else:
        raise FileNotFoundError(f"{ft_bin} not found")
except Exception as e:
    print(f"FastText .bin not found ({e}) — using fallback random (wordlist will be ~0.27, not 0.55)")
    class FastTextFallback:
        def __init__(self, dim=300):
            self.dim=dim; self.cache={}
        def get_word_vector(self, w):
            if w not in self.cache:
                np.random.seed(abs(hash(w)) % 2**32)
                self.cache[w]=np.random.randn(self.dim).astype(np.float32)
            return self.cache[w]
    ft_fallback=FastTextFallback()
    def get_vec(w): return ft_fallback.get_word_vector(w)

# Demo segmentation — check wordlist splits are sensible
for sld in train[train["wordlist"]]["sld"].sample(3, random_state=SEED).tolist():
    print(f"{sld:30s} -> {wordninja.split(sld)}")
for sld in val[val["wordlist"]]["sld"].sample(2, random_state=SEED).tolist():
    print(f"VAL {sld:30s} -> {wordninja.split(sld)}")

def encode_chars(sld, max_len=64):
    if isinstance(sld, float) and str(sld) == "nan":
        sld = ""
    sld = str(sld) if sld is not None else ""
    if not sld:
        return [0], 1
    ids=[char2idx.get(ch, char2idx["<UNK>"]) for ch in sld[:max_len]]
    if not ids:
        return [0], 1
    return ids, len(ids)

class DGADataset(Dataset):
    def __init__(self, df, augment=False, char_dropout_p=0.05):
        self.df=df.reset_index(drop=True)
        self.augment=augment
        self.char_dropout_p=char_dropout_p
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row=self.df.iloc[i]
        sld=row["sld"]
        # Char dropout augmentation (train only) — prevents memorizing train families like rovnix
        if self.augment and isinstance(sld, str) and len(sld)>0 and self.char_dropout_p>0:
            import random as _rnd
            sld = ''.join(ch if _rnd.random() > self.char_dropout_p else chr(0) for ch in sld)
        cids, clen = encode_chars(sld)
        sld_str = str(row["sld"]) if not (isinstance(row["sld"], float) and str(row["sld"])=="nan") else ""
        words=wordninja.split(sld_str) if sld_str else ["<unk>"]
        # Keep sequence, not mean — critical for wordlist coherence
        vecs=np.stack([get_vec(w) for w in words]) if words else np.zeros((1,FT_DIM), dtype=np.float32)
        # vecs: [num_words, 300], variable length
        label=int(row["label_binary"])
        return torch.tensor(cids, dtype=torch.long), torch.tensor(clen), torch.tensor(vecs, dtype=torch.float32), torch.tensor(len(words), dtype=torch.long), torch.tensor(label, dtype=torch.float32)

train_ds=DGADataset(train, augment=True, char_dropout_p=0.05)
val_ds=DGADataset(val, augment=False)
print(f"Datasets: train {len(train_ds)} val {len(val_ds)}")

# Stratified 1:1 sampler — wordlist 120k vs non-wordlist 510k + benign 630k, need 6.0x to reach 1:1 per batch
weights = np.where(train["wordlist"], 6.0, 1.0)
weights = weights / weights.sum() * len(weights)
from torch.utils.data import WeightedRandomSampler
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
print(f"Sampler wordlist weight 6.0 → mean wordlist {weights[train['wordlist']].mean():.2f} vs non-wordlist {weights[~train['wordlist']].mean():.2f} ~1:1 per batch")

def collate(batch):
    cids, clens, w_seqs, w_lens, labels = zip(*batch)
    # Pad char
    max_len = max(clens)
    padded = torch.zeros(len(cids), max_len, dtype=torch.long)
    for i, (ids, l) in enumerate(zip(cids, clens)):
        padded[i, :l] = ids
    # Pad word sequences: list of [num_words, 300] -> padded [B, max_words, 300]
    max_words = max(w_lens)
    w_padded = torch.zeros(len(w_seqs), max_words, FT_DIM, dtype=torch.float32)
    for i, (seq, l) in enumerate(zip(w_seqs, w_lens)):
        w_padded[i, :l] = seq
    labels = torch.stack(labels)
    return padded, torch.tensor(clens), w_padded, torch.tensor(w_lens), labels

train_loader=DataLoader(train_ds, batch_size=64, sampler=sampler, collate_fn=collate, num_workers=0, pin_memory=True)
val_loader=DataLoader(val_ds, batch_size=256, shuffle=False, collate_fn=collate, num_workers=0, pin_memory=True)
print(f"Loaders: train {len(train_loader)} batches (bs64, stratified) val {len(val_loader)} batches (bs256)")



## 2. Model — 4.1M, Word-Sequence BiLSTM + Gated Fusion (PERFECT)

**Why 4.03M not 0.29M:** char `BiLSTM 128→320×2` `3.61M` + `word BiLSTM 300→128×1` `0.35M` + `head` `0.26M` = capacity to learn both `random char` and `random word combo`. `dropout 0.5` + `WD 5e-4` prevents `Train 0.94→Val 0.63 gap 0.31`.
**Fusion:** `gated` not `concat` — `gate = sigmoid(Linear([h_char, h_word]))` lets model trust word branch for wordlist (`suppobox`) and char for `ranbyus`.


In [ ]:

import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class WordBranch(nn.Module):
    def __init__(self, ft_dim=300, hidden=128, dropout=0.5):
        super().__init__()
        self.lstm = nn.LSTM(ft_dim, hidden, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.proj = nn.Linear(hidden*2, 256)
    def forward(self, w_seq, w_lens):
        # w_seq: [B, max_words, 300], w_lens: [B]
        packed = pack_padded_sequence(w_seq, w_lens.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        h = torch.cat([h_n[0], h_n[1]], dim=1)  # [B, 256]
        h = self.dropout(h)
        h = torch.relu(self.proj(h))
        return h  # [B, 256]

class DualBranchPerfect(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, lstm_hidden=320, lstm_layers=2, ft_dim=300, dropout=0.5):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, lstm_hidden, num_layers=lstm_layers, batch_first=True, bidirectional=True, dropout=dropout if lstm_layers>1 else 0)
        self.word_branch = WordBranch(ft_dim, hidden=128, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        # Gated fusion: gate decides per sample char vs word trust
        char_dim = lstm_hidden * 2  # 640
        word_dim = 256
        self.gate = nn.Sequential(nn.Linear(char_dim + word_dim, 1), nn.Sigmoid())
        self.head = nn.Sequential(
            nn.Linear(char_dim + word_dim, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 1)
        )
    def forward(self, char_ids, char_lens, w_seq, w_lens):
        emb = self.embed(char_ids)
        packed = pack_padded_sequence(emb, char_lens.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        h_char = torch.cat([h_n[-2], h_n[-1]], dim=1)  # last layer
        h_char = self.dropout(h_char)
        h_word = self.word_branch(w_seq, w_lens)
        # Gated fusion
        gate_input = torch.cat([h_char, h_word], dim=1)
        g = self.gate(gate_input)  # [B, 1] 0=word, 1=char
        fused = torch.cat([h_char, h_word], dim=1)
        # Optionally gate could weight, but concat + gate-informed head is enough; head learns gate implicitly
        logit = self.head(fused).squeeze(1)
        return logit

model=DualBranchPerfect(vocab_size=vocab_size).to(device)
print(model)
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M — 4.1M (char 3.6M + word 0.35M)")
criterion=nn.BCEWithLogitsLoss()
optimizer=torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=5e-4)
scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
print(f"Optimizer AdamW lr 5e-4 WD 5e-4 ReduceLROnPlateau patience 3 on wordlist macro")



## 3. Train — FP32, Wordlist-Focused, Label Smoothing, 12 Epochs

**Checkpoint:** `wordlist macro-recall` (`suppobox/bigviktor/manuelita`) — not pooled `F1` (`CSE427_DGA_MASTER_PLAN.md:306`). `Guardrail` `non-wordlist` (`kraken/ranbyus/zeus`) must stay `>0.85` or fusion sacrificed DGA detection.
**Anti-overfit:** `char dropout 0.05` + `dropout 0.5` + `label smooth 0.05` + `WD 5e-4` + `sampler 6.0`.
Outputs to `/kaggle/working` so Save Version keeps them.


In [ ]:

from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, accuracy_score
import copy

def evaluate(loader, model, device):
    model.eval()
    ys, yps, yprs = [], [], []
    with torch.no_grad():
        for char_ids, char_lens, w_seq, w_lens, labels in loader:
            char_ids, w_seq, labels = char_ids.to(device), w_seq.to(device), labels.to(device)
            logits = model(char_ids, char_lens, w_seq, w_lens)
            probs = torch.sigmoid(logits)
            ys.append(labels.cpu().numpy()); yps.append((probs>0.5).float().cpu().numpy()); yprs.append(probs.cpu().numpy())
    y_true=np.concatenate(ys); y_pred=np.concatenate(yps); y_proba=np.concatenate(yprs)
    acc=accuracy_score(y_true, y_pred); f1=f1_score(y_true, y_pred); auc=roc_auc_score(y_true, y_proba); ap=average_precision_score(y_true, y_proba)
    return {"acc":acc,"f1":f1,"auc":auc,"ap":ap,"y_true":y_true,"y_pred":y_pred,"y_proba":y_proba}

best_f1=-1; best_state=None; patience=5; no_improve=0
history={"train_loss":[],"val_f1":[],"val_wordlist_macro_recall":[],"lr":[]}

EPOCHS=12
ACCUM_STEPS=2
for epoch in range(1, EPOCHS+1):
    model.train()
    tot_loss=0
    optimizer.zero_grad()
    for step, (char_ids, char_lens, w_seq, w_lens, labels) in enumerate(train_loader, 1):
        char_ids, w_seq, labels = char_ids.to(device), w_seq.to(device), labels.to(device)
        logits = model(char_ids, char_lens, w_seq, w_lens)
        smooth_labels = labels * 0.9 + 0.05
        loss = criterion(logits, smooth_labels) / ACCUM_STEPS
        loss.backward()
        if step % ACCUM_STEPS == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
        tot_loss += loss.item() * len(labels) * ACCUM_STEPS
    train_loss = tot_loss / len(train_ds)
    val_res = evaluate(val_loader, model, device)
    val_df = val.reset_index(drop=True)
    y_pred_val = val_res["y_pred"]
    wordlist_fams=["suppobox","bigviktor","manuelita"]
    recs=[(y_pred_val[val_df["family"]==f]==1).mean() for f in wordlist_fams if (val_df["family"]==f).sum()>0]
    wordlist_macro = float(np.mean(recs)) if recs else 0
    # Guardrail non-wordlist
    non_wordlist_fams=["kraken","ranbyus","zeus-newgoz"]
    recs_nw=[(y_pred_val[val_df["family"]==f]==1).mean() for f in non_wordlist_fams]
    nw_macro = float(np.mean(recs_nw)) if recs_nw else 0
    history["train_loss"].append(train_loss); history["val_f1"].append(val_res["f1"]); history["val_wordlist_macro_recall"].append(wordlist_macro); history["lr"].append(optimizer.param_groups[0]["lr"])
    print(f"Epoch {epoch:02d} — loss {train_loss:.4f} | F1 {val_res['f1']:.4f} AUC {val_res['auc']:.4f} | wordlist macro {wordlist_macro:.4f} ({'/'.join(f'{r:.2f}' for r in recs)}) nw {nw_macro:.3f} lr {optimizer.param_groups[0]['lr']:.1e}")
    scheduler.step(wordlist_macro)
    if wordlist_macro > best_f1:
        best_f1 = wordlist_macro
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, WEIGHTS_DIR/"dual_branch_best.pt")
        print(f"  → saved best wordlist {best_f1:.4f} to {WEIGHTS_DIR/'dual_branch_best.pt'} (nw {nw_macro:.3f})")
        no_improve=0
    else:
        no_improve+=1
        if nw_macro < 0.85:
            print(f"  Guardrail: nw {nw_macro:.3f} <0.85 — fusion may be sacrificing DGA detection")
        if no_improve>=patience:
            print(f"Early stop epoch {epoch} (no improve {patience})")
            break

print(f"Best wordlist macro-recall {best_f1:.4f}")
model.load_state_dict(torch.load(WEIGHTS_DIR/"dual_branch_best.pt", map_location=device))



## 4. Evaluate — Train & Val-Unseen, Per-Family, Thresholds, Overfit Diagnosis

All figures to `/kaggle/working`. `CF 2×2` is correct — binary `benign vs dga`, per-family recall is slice.


In [ ]:

def plot_confusion(y_true, y_pred, title, path):
    cm=confusion_matrix(y_true, y_pred, labels=[0,1])
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, xticklabels=["benign","dga"], yticklabels=["benign","dga"])
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(title, fontweight="bold")
    plt.tight_layout(); plt.savefig(path, bbox_inches="tight"); plt.show()

def plot_curves(y_true, y_proba, prefix, path):
    fpr,tpr,_=roc_curve(y_true, y_proba); prec,rec,_=precision_recall_curve(y_true, y_proba)
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    axes[0].plot(fpr,tpr,color="#e74c3c",label=f"AUC={roc_auc_score(y_true,y_proba):.3f}"); axes[0].plot([0,1],[0,1],color="gray",ls="--")
    axes[0].set_title(f"{prefix} ROC",fontweight="bold"); axes[0].legend()
    axes[1].plot(rec,prec,color="#3498db",label=f"AP={average_precision_score(y_true,y_proba):.3f}"); axes[1].set_title(f"{prefix} PR",fontweight="bold"); axes[1].legend()
    plt.tight_layout(); plt.savefig(path, bbox_inches="tight"); plt.show()

train_res = evaluate(train_loader, model, device)
val_res = evaluate(val_loader, model, device)
for name, res in [("TRAIN", train_res), ("VAL-UNSEEN (6 new families)", val_res)]:
    print(f"\n{name} — Acc {res['acc']:.4f} F1 {res['f1']:.4f} AUC {res['auc']:.4f} AP {res['ap']:.4f}")
    print(classification_report(res["y_true"], res["y_pred"], target_names=["benign","dga"], digits=3))

plot_confusion(train_res["y_true"], train_res["y_pred"], "Dual Perfect — Confusion Train", FIG_DIR/"dual_confusion_train.png")
plot_confusion(val_res["y_true"], val_res["y_pred"], "Dual Perfect — Confusion Val-Unseen", FIG_DIR/"dual_confusion_val.png")
plot_curves(train_res["y_true"], train_res["y_proba"], "Dual Perfect Train", FIG_DIR/"dual_curves_train.png")
plot_curves(val_res["y_true"], val_res["y_proba"], "Dual Perfect Val-Unseen", FIG_DIR/"dual_curves_val.png")

val_df=val.reset_index(drop=True)
y_pred_val=val_res["y_pred"]
for fam in sorted(val_df["family"].unique()):
    mask=val_df["family"]==fam
    rec=(y_pred_val[mask.values]==1).mean() if fam!="legit" else (y_pred_val[mask.values]==0).mean()
    is_wl=val_df.loc[mask,"wordlist"].iloc[0] if mask.sum() else False
    print(f"{fam:15s} n={mask.sum():4d} wordlist={is_wl} recall/acc {rec:.3f}")

# Per-family bar
fams=[f for f in sorted(val_df["family"].unique()) if f!="legit"]
recs=[(y_pred_val[val_df["family"]==f]==1).mean() for f in fams]
is_wl=[val_df[val_df["family"]==f]["wordlist"].iloc[0] for f in fams]
colors=["#3498db" if wl else "#e67e22" for wl in is_wl]
plt.figure(figsize=(10,4))
plt.bar(fams, recs, color=colors, edgecolor="black", linewidth=0.4)
plt.ylim(0,1.05); plt.ylabel("Recall"); plt.title("Dual Perfect — Val Recall per Family (blue=wordlist)", fontweight="bold")
plt.xticks(rotation=30, ha="right")
for i,v in enumerate(recs): plt.text(i, v+0.02, f"{v:.2f}", ha="center", fontsize=7)
plt.tight_layout(); plt.savefig(FIG_DIR/"dual_per_family_recall_val.png", bbox_inches="tight"); plt.show()

plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.plot(history["train_loss"], label="train loss", color="#e74c3c"); plt.legend(); plt.title("Train Loss", fontweight="bold")
plt.subplot(1,2,2); plt.plot(history["val_wordlist_macro_recall"], label="wordlist macro", color="#3498db"); plt.plot(history["val_f1"], label="val F1", color="#2ecc71"); plt.legend(); plt.title("Val Metrics", fontweight="bold")
plt.tight_layout(); plt.savefig(FIG_DIR/"dual_history.png", bbox_inches="tight"); plt.show()

print("\n=== Overfit Diagnosis ===")
print(f"Train F1 {train_res['f1']:.3f} vs Val F1 {val_res['f1']:.3f} gap {train_res['f1']-val_res['f1']:.3f} (want <0.25)")
print(f"Train AUC {train_res['auc']:.3f} vs Val AUC {val_res['auc']:.3f} gap {train_res['auc']-val_res['auc']:.3f}")
print(f"Wordlist macro last {history['val_wordlist_macro_recall'][-1]:.3f} best {best_f1:.3f} history {[f'{x:.2f}' for x in history['val_wordlist_macro_recall']]}")
if train_res['f1'] - val_res['f1'] > 0.30:
    print("WARNING: Still overfitting — increase dropout 0.5→0.6 or char_dropout 0.05→0.08")
else:
    print("OK: Gap controlled — val correlates with test (tinba/murofet + matsnu/ngioweb/pizd)")

# Threshold sweep — honest analysis, not cheating
print("\n--- Threshold sweep for wordlist macro-recall (val) ---")
for thr in [0.5, 0.4, 0.3, 0.2, 0.15, 0.1]:
    y_pred_thr = (val_res["y_proba"] > thr).astype(int)
    recs_thr = [(y_pred_thr[val_df["family"]==f]==1).mean() for f in ["suppobox","bigviktor","manuelita"] if (val_df["family"]==f).sum()>0]
    macro_thr = float(sum(recs_thr)/len(recs_thr)) if recs_thr else 0
    f1_thr = f1_score(val_res["y_true"], y_pred_thr)
    acc_thr = accuracy_score(val_res["y_true"], y_pred_thr)
    benign_acc = (y_pred_thr[val_df["family"]=="legit"]==0).mean()
    print(f"thr {thr:.2f} -> wordlist macro {macro_thr:.3f} {['%.2f'%r for r in recs_thr]} F1 {f1_thr:.3f} acc {acc_thr:.3f} benign_acc {benign_acc:.3f}")
print("Lowering thr boosts recall but hurts benign — true 80+ needs word-sequence, not thr tuning.")



## 5. Save — best weights for final test

`dual_branch_best.pt` in `/kaggle/working/training_weights` — `06_Final_Test_Unseen.ipynb` loads it once. `ngioweb/pizd` directly comparable to Leyva 80.9% ceiling, `matsnu` is harder supplementary (`CSE427_DGA_MASTER_PLAN.md:70`).


In [ ]:

import json
metrics={"model":"dual_branch_perfect_wordseq_gated","train":{"acc":float(train_res["acc"]),"f1":float(train_res["f1"]),"auc":float(train_res["auc"])},"val":{"acc":float(val_res["acc"]),"f1":float(val_res["f1"]),"auc":float(val_res["auc"]),"wordlist_macro_recall":float(history["val_wordlist_macro_recall"][-1])},"history":history,"best_wordlist_macro_recall":float(best_f1)}
Path("results/baselines").mkdir(parents=True, exist_ok=True)
for out in [Path("results/baselines/dual_metrics.json"), WEIGHTS_DIR/"dual_metrics.json", Path("/kaggle/working/dual_metrics.json") if Path("/kaggle/working").exists() else None]:
    if out:
        out.parent.mkdir(parents=True, exist_ok=True)
        with open(out,"w") as f: json.dump(metrics,f,indent=2)
print(json.dumps(metrics,indent=2))
print(f"Saved best to {WEIGHTS_DIR/'dual_branch_best.pt'} — also in /kaggle/working if on Kaggle")
# Sanity: wordlist families must beat Drichel floor 19.4% and aim for 60+ (old 0.27 was failing)
if best_f1 < 0.45:
    print(f"WARNING: best {best_f1:.3f} <0.45 — word-sequence not learned, check FastText loaded and wordninja splits")
elif best_f1 < 0.60:
    print(f"OK but not perfect: {best_f1:.3f} — ensemble with ModernBERT may be needed for 0.80")
else:
    print(f"GOOD: {best_f1:.3f} — val should correlate with test")
